# Kerr Black Hole — Full Tensor Analysis

The Kerr solution describes a rotating black hole. This tutorial runs the complete
tensor computation pipeline and verifies physical properties:

- Christoffel symbols
- Riemann tensor and antisymmetry check
- Vacuum solution verification ($R_{ab} = 0$, $G_{ab} = 0$)

In [1]:
import os
from symbolica import set_license_key

_key = os.environ.get("SYMBOLICA_LICENSE", "")
if _key:
    set_license_key(_key)

from gravica import (
    ChristoffelSymbols,
    RiemannTensor,
    RicciTensor,
    EinsteinTensor,
    ricci_scalar,
    display,
    check,
)
from gravica.simplify import simplify, str_is_zero
from gravica.metrics import kerr

## 1. The Kerr Metric

Boyer-Lindquist coordinates $(t, r, \theta, \varphi)$ with mass $M$ and spin parameter $a = J/M$.

$$
\Sigma = r^2 + a^2\cos^2\theta, \quad \Delta = r^2 - 2Mr + a^2
$$

The off-diagonal $g_{t\varphi}$ component reflects **frame dragging**.

In [2]:
g = kerr()
coords = ["t", "r", "\\theta", "\\varphi"]

print(f"Coordinates: {g.coords}")
print(f"Diagonal:    {g.is_diagonal()}")
print()
display.components_table(display.nonzero_components(g, coords), g)

Coordinates: (t, r, theta, phi)
Diagonal:    False

┌────────────────────────────────────────────────────────┐
│ You are running a restricted Symbolica instance.       │
│                                                        │
│ This mode is only permitted for non-commercial use and │
│ is limited to one instance and core per machine.       │
│                                                        │
│ Hobbyists can easily acquire a free license key        │
│ that unlocks all cores and removes this banner:        │
│                                                        │
│   from symbolica import *                              │
│   request_hobbyist_license('YOUR_NAME', 'YOUR_EMAIL')  │
│                                                        │
│ All other users can obtain a free 30-day trial key:    │
│                                                        │
│   from symbolica import *                              │
│   request_trial_license('NAME', 'EMAIL', 'EMPLOYER')   │
│   

| Component | Value |
|-----------|-------|
| $g_{t t}$ | $\frac{-2 r M+r^{2}+a^{2} \cos\!\left(\theta\right)^{2}}{r^{2}+a^{2} \cos\!\left(\theta\right)^{2}}$ |
| $g_{t \varphi}$ | $\frac{-2 r M a \sin\!\left(\theta\right)^{2}}{r^{2}+a^{2} \cos\!\left(\theta\right)^{2}}$ |
| $g_{r r}$ | $\frac{-r^{2}-a^{2} \cos\!\left(\theta\right)^{2}}{-2 r M+r^{2}+a^{2}}$ |
| $g_{\theta \theta}$ | $-r^{2}-a^{2} \cos\!\left(\theta\right)^{2}$ |
| $g_{\varphi \varphi}$ | $\frac{-2 r M a^{2} \sin\!\left(\theta\right)^{4}-r^{2} a^{2} \sin\!\left(\theta\right)^{2}-r^{2} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-r^{4} \sin\!\left(\theta\right)^{2}-a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}}{r^{2}+a^{2} \cos\!\left(\theta\right)^{2}}$ |

## 2. Tensor Pipeline

In [3]:
christoffel = ChristoffelSymbols(g)
riemann = RiemannTensor(christoffel)
ricci = RicciTensor(riemann)
einstein = EinsteinTensor(ricci)

## 3. Christoffel Symbols

The Kerr metric is non-diagonal, so it has more non-zero Christoffel symbols than Schwarzschild.

In [4]:
christoffel_items = display.nonzero_components(christoffel, coords)
print(f"Number of non-zero Christoffel symbols: {len(christoffel_items)}")
display.components_table(christoffel_items, christoffel)

Number of non-zero Christoffel symbols: 20


| Component | Value |
|-----------|-------|
| $\Gamma^{t}_{t r}$ | $\frac{-M a^{4} \cos\!\left(\theta\right)^{2}+r^{2} M a^{2}-r^{2} M a^{2} \cos\!\left(\theta\right)^{2}+r^{4} M}{2 r M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r M a^{4} \cos\!\left(\theta\right)^{2}+2 r^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{2}+2 r^{3} M a^{2} \sin\!\left(\theta\right)^{2}-2 r^{3} M a^{2} \cos\!\left(\theta\right)^{2}+r^{4} a^{2}+2 r^{4} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M+r^{6}+a^{6} \cos\!\left(\theta\right)^{4}}$ |
| $\Gamma^{t}_{t \theta}$ | $\frac{-2 r M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+4 r^{2} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-2 r^{3} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{2 r M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r M a^{4} \cos\!\left(\theta\right)^{2}+2 r^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{2}+2 r^{3} M a^{2} \sin\!\left(\theta\right)^{2}-2 r^{3} M a^{2} \cos\!\left(\theta\right)^{2}+r^{4} a^{2}+2 r^{4} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M+r^{6}+a^{6} \cos\!\left(\theta\right)^{4}}$ |
| $\Gamma^{t}_{r \varphi}$ | $\frac{-M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{2} M a^{3} \sin\!\left(\theta\right)^{2}+r^{2} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{4} M a \sin\!\left(\theta\right)^{2}}{2 r M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r M a^{4} \cos\!\left(\theta\right)^{2}+2 r^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{2}+2 r^{3} M a^{2} \sin\!\left(\theta\right)^{2}-2 r^{3} M a^{2} \cos\!\left(\theta\right)^{2}+r^{4} a^{2}+2 r^{4} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M+r^{6}+a^{6} \cos\!\left(\theta\right)^{4}}$ |
| $\Gamma^{t}_{\theta \varphi}$ | $\frac{-2 r M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{2} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-2 r^{3} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)}{2 r M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r M a^{4} \cos\!\left(\theta\right)^{2}+2 r^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{2}+2 r^{3} M a^{2} \sin\!\left(\theta\right)^{2}-2 r^{3} M a^{2} \cos\!\left(\theta\right)^{2}+r^{4} a^{2}+2 r^{4} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M+r^{6}+a^{6} \cos\!\left(\theta\right)^{4}}$ |
| $\Gamma^{r}_{t t}$ | $\frac{2 r M^{2} a^{2} \cos\!\left(\theta\right)^{2}-M a^{4} \cos\!\left(\theta\right)^{2}+r^{2} M a^{2}-r^{2} M a^{2} \cos\!\left(\theta\right)^{2}-2 r^{3} M^{2}+r^{4} M}{3 r^{2} a^{4} \cos\!\left(\theta\right)^{4}+3 r^{4} a^{2} \cos\!\left(\theta\right)^{2}+r^{6}+a^{6} \cos\!\left(\theta\right)^{6}}$ |
| $\Gamma^{r}_{t \varphi}$ | $\frac{2 r M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{2} M a^{3} \sin\!\left(\theta\right)^{2}-r^{2} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{3} M^{2} a \sin\!\left(\theta\right)^{2}+r^{4} M a \sin\!\left(\theta\right)^{2}}{3 r^{2} a^{4} \cos\!\left(\theta\right)^{4}+3 r^{4} a^{2} \cos\!\left(\theta\right)^{2}+r^{6}+a^{6} \cos\!\left(\theta\right)^{6}}$ |
| $\Gamma^{r}_{r r}$ | $\frac{r a^{2}-r a^{2} \cos\!\left(\theta\right)^{2}+M a^{2} \cos\!\left(\theta\right)^{2}-r^{2} M}{-2 r M a^{2} \cos\!\left(\theta\right)^{2}+r^{2} a^{2}+r^{2} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{3} M+r^{4}+a^{4} \cos\!\left(\theta\right)^{2}}$ |
| $\Gamma^{r}_{r \theta}$ | $\frac{-a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{r^{2}+a^{2} \cos\!\left(\theta\right)^{2}}$ |
| $\Gamma^{r}_{\theta \theta}$ | $\frac{-r a^{2}+2 r^{2} M-r^{3}}{r^{2}+a^{2} \cos\!\left(\theta\right)^{2}}$ |
| $\Gamma^{r}_{\varphi \varphi}$ | $\frac{2 r M^{2} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-r a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{2} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r^{2} M a^{4} \sin\!\left(\theta\right)^{4}-r^{2} M a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-2 r^{3} M^{2} a^{2} \sin\!\left(\theta\right)^{4}-2 r^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-r^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{4} M a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{4} M a^{2} \sin\!\left(\theta\right)^{4}-r^{5} a^{2} \sin\!\left(\theta\right)^{2}-2 r^{5} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+2 r^{6} M \sin\!\left(\theta\right)^{2}-r^{7} \sin\!\left(\theta\right)^{2}}{3 r^{2} a^{4} \cos\!\left(\theta\right)^{4}+3 r^{4} a^{2} \cos\!\left(\theta\right)^{2}+r^{6}+a^{6} \cos\!\left(\theta\right)^{6}}$ |
| $\Gamma^{\theta}_{t t}$ | $\frac{-2 r M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{3 r^{2} a^{4} \cos\!\left(\theta\right)^{4}+3 r^{4} a^{2} \cos\!\left(\theta\right)^{2}+r^{6}+a^{6} \cos\!\left(\theta\right)^{6}}$ |
| $\Gamma^{\theta}_{t \varphi}$ | $\frac{-2 r M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-2 r M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-2 r^{3} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{3 r^{2} a^{4} \cos\!\left(\theta\right)^{4}+3 r^{4} a^{2} \cos\!\left(\theta\right)^{2}+r^{6}+a^{6} \cos\!\left(\theta\right)^{6}}$ |
| $\Gamma^{\theta}_{r r}$ | $\frac{a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{-2 r M a^{2} \cos\!\left(\theta\right)^{2}+r^{2} a^{2}+r^{2} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{3} M+r^{4}+a^{4} \cos\!\left(\theta\right)^{2}}$ |
| $\Gamma^{\theta}_{r \theta}$ | $\frac{r}{r^{2}+a^{2} \cos\!\left(\theta\right)^{2}}$ |
| $\Gamma^{\theta}_{\theta \theta}$ | $\frac{-a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{r^{2}+a^{2} \cos\!\left(\theta\right)^{2}}$ |
| $\Gamma^{\theta}_{\varphi \varphi}$ | $\frac{-4 r M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-2 r M a^{4} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-2 r^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-4 r^{3} M a^{2} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-r^{4} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-2 r^{4} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}}{3 r^{2} a^{4} \cos\!\left(\theta\right)^{4}+3 r^{4} a^{2} \cos\!\left(\theta\right)^{2}+r^{6}+a^{6} \cos\!\left(\theta\right)^{6}}$ |
| $\Gamma^{\varphi}_{t r}$ | $\frac{M a^{3} \cos\!\left(\theta\right)^{2}-r^{2} M a}{2 r M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r M a^{4} \cos\!\left(\theta\right)^{2}+2 r^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{2}+2 r^{3} M a^{2} \sin\!\left(\theta\right)^{2}-2 r^{3} M a^{2} \cos\!\left(\theta\right)^{2}+r^{4} a^{2}+2 r^{4} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M+r^{6}+a^{6} \cos\!\left(\theta\right)^{4}}$ |
| $\Gamma^{\varphi}_{t \theta}$ | $\frac{r M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+r M a^{3} \cos\!\left(\theta\right)^{3}-2 r^{2} M^{2} a \cos\!\left(\theta\right)+r^{3} M a \cos\!\left(\theta\right)}{-r M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+r M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+\frac{1}{2} r^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-r^{3} M a^{2} \sin\!\left(\theta\right)-r^{3} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+r^{3} M a^{2} \sin\!\left(\theta\right)^{3}+\frac{1}{2} r^{4} a^{2} \sin\!\left(\theta\right)+r^{4} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-r^{5} M \sin\!\left(\theta\right)+\frac{1}{2} r^{6} \sin\!\left(\theta\right)+\frac{1}{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}}$ |
| $\Gamma^{\varphi}_{r \varphi}$ | $\frac{r a^{4} \cos\!\left(\theta\right)^{4}+M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-r^{2} M a^{2} \sin\!\left(\theta\right)^{2}-2 r^{2} M a^{2} \cos\!\left(\theta\right)^{2}+2 r^{3} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{4} M+r^{5}}{2 r M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r M a^{4} \cos\!\left(\theta\right)^{2}+2 r^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{2}+2 r^{3} M a^{2} \sin\!\left(\theta\right)^{2}-2 r^{3} M a^{2} \cos\!\left(\theta\right)^{2}+r^{4} a^{2}+2 r^{4} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M+r^{6}+a^{6} \cos\!\left(\theta\right)^{4}}$ |
| $\Gamma^{\varphi}_{\theta \varphi}$ | $\frac{4 r M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+2 r M a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-2 r M a^{4} \cos\!\left(\theta\right)^{3}-4 r^{2} M^{2} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+2 r^{2} a^{4} \cos\!\left(\theta\right)^{3}+r^{2} a^{4} \cos\!\left(\theta\right)^{5}-2 r^{3} M a^{2} \cos\!\left(\theta\right)+4 r^{3} M a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-2 r^{3} M a^{2} \cos\!\left(\theta\right)^{3}+r^{4} a^{2} \cos\!\left(\theta\right)+2 r^{4} a^{2} \cos\!\left(\theta\right)^{3}-2 r^{5} M \cos\!\left(\theta\right)+r^{6} \cos\!\left(\theta\right)+a^{6} \cos\!\left(\theta\right)^{5}}{-2 r M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+2 r M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+2 r^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{2} \sin\!\left(\theta\right)-2 r^{3} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+2 r^{3} M a^{2} \sin\!\left(\theta\right)^{3}+r^{4} a^{2} \sin\!\left(\theta\right)+2 r^{4} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-2 r^{5} M \sin\!\left(\theta\right)+r^{6} \sin\!\left(\theta\right)+a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}}$ |

## 4. Vacuum Solution Verification

The Kerr solution satisfies the vacuum Einstein equations ($T_{ab} = 0$).
We verify $R_{ab} = 0$, $R = 0$, and $G_{ab} = 0$.

In [5]:
from IPython.display import Markdown

ricci_is_zero = check.zero(ricci, simplify)
R = simplify(ricci_scalar(ricci))
scalar_is_zero = str_is_zero(R)
einstein_is_zero = check.zero(einstein, simplify)

print(f"R_ab = 0:  {ricci_is_zero}")
print(f"R = {R}  (zero: {scalar_is_zero})")
print(f"G_ab = 0:  {einstein_is_zero}")

all_ok = ricci_is_zero and scalar_is_zero and einstein_is_zero
Markdown(f"**Vacuum solution check**: **{'All verified' if all_ok else 'FAILED'}**")

R_ab = 0:  False
R = (-8*r*M*a^10*sin(theta)^2*cos(theta)^4+10*r*M*a^10*sin(theta)^2*cos(theta)^6+4*r*M*a^10*sin(theta)^4*cos(theta)^4+4*r*M*a^10*cos(theta)^4-10*r*M*a^10*cos(theta)^6+6*r*M*a^10*cos(theta)^8+32*r^2*M^2*a^8*sin(theta)^2*cos(theta)^2-30*r^2*M^2*a^8*sin(theta)^2*cos(theta)^4+10*r^2*M^2*a^8*sin(theta)^2*cos(theta)^6-28*r^2*M^2*a^8*sin(theta)^4*cos(theta)^2+18*r^2*M^2*a^8*sin(theta)^4*cos(theta)^4+8*r^2*M^2*a^8*sin(theta)^6*cos(theta)^2-12*r^2*M^2*a^8*cos(theta)^2+12*r^2*M^2*a^8*cos(theta)^4-2*r^2*a^10*sin(theta)^2*cos(theta)^4-4*r^2*a^10*sin(theta)^2*cos(theta)^6+2*r^2*a^10*cos(theta)^4+2*r^2*a^10*cos(theta)^6-4*r^2*a^10*cos(theta)^8-16*r^3*M*a^8*sin(theta)^2*cos(theta)^2+6*r^3*M*a^8*sin(theta)^2*cos(theta)^4+14*r^3*M*a^8*sin(theta)^2*cos(theta)^6+8*r^3*M*a^8*sin(theta)^4*cos(theta)^2+4*r^3*M*a^8*sin(theta)^4*cos(theta)^4+8*r^3*M*a^8*cos(theta)^2-10*r^3*M*a^8*cos(theta)^4-8*r^3*M*a^8*cos(theta)^6+10*r^3*M*a^8*cos(theta)^8-32*r^3*M^3*a^6*sin(theta)^2*cos(theta)^2+16*r^3*M^3

**Vacuum solution check**: **FAILED**

## 5. Riemann Tensor Antisymmetry

$$R^a{}_{bcd} + R^a{}_{bdc} = 0$$

In [6]:
antisym_ok = check.antisymmetry(riemann, simplify)
print(f"Antisymmetry R^a_bcd + R^a_bdc = 0: {'Verified' if antisym_ok else 'FAILED'}")

Antisymmetry R^a_bcd + R^a_bdc = 0: Verified


## 6. Riemann Tensor Components

In [7]:
riemann_items = display.nonzero_components(riemann, coords)
print(f"Non-zero independent components (c < d): {len(riemann_items)}")
display.components_table(riemann_items, riemann)

Non-zero independent components (c < d): 44


| Component | Value |
|-----------|-------|
| $R^{t}{}_{t t \varphi}$ | $\frac{3 r^{2} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-6 r^{3} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-r^{4} M^{2} a^{3} \sin\!\left(\theta\right)^{2}+3 r^{4} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+2 r^{5} M^{3} a \sin\!\left(\theta\right)^{2}-r^{6} M^{2} a \sin\!\left(\theta\right)^{2}}{r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-r M a^{8} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{2} a^{8} \cos\!\left(\theta\right)^{8}+3 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{3} M a^{6} \cos\!\left(\theta\right)^{4}-r^{3} M a^{6} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{6} \cos\!\left(\theta\right)^{4}+2 r^{4} a^{6} \cos\!\left(\theta\right)^{6}+3 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{6} a^{4} \cos\!\left(\theta\right)^{2}+3 r^{6} a^{4} \cos\!\left(\theta\right)^{4}-r^{7} M a^{2}+r^{7} M a^{2} \sin\!\left(\theta\right)^{2}-3 r^{7} M a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{2} r^{8} a^{2}+2 r^{8} a^{2} \cos\!\left(\theta\right)^{2}-r^{9} M+\frac{1}{2} r^{10}+\frac{1}{2} a^{10} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{t r \theta}$ | $\frac{\frac{1}{2} r M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-r^{2} M^{3} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-\frac{1}{2} r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-\frac{1}{2} r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-2 r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+3 r^{6} M^{3} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-\frac{3}{2} r^{7} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-r M a^{10} \cos\!\left(\theta\right)^{6}-2 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+r^{2} a^{10} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{2} a^{10} \cos\!\left(\theta\right)^{8}+3 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-3 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+2 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+\frac{3}{2} r^{4} a^{8} \cos\!\left(\theta\right)^{4}+2 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{4} r^{4} a^{8} \cos\!\left(\theta\right)^{8}+3 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-6 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+r^{6} M^{2} a^{4}-2 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+r^{6} a^{6} \cos\!\left(\theta\right)^{2}+3 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+r^{6} a^{6} \cos\!\left(\theta\right)^{6}-r^{7} M a^{4}+r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+3 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-6 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-3 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{8} M^{2} a^{2}-2 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+2 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{4} r^{8} a^{4}+2 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+\frac{3}{2} r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-3 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} M^{2}+\frac{1}{2} r^{10} a^{2}+r^{10} a^{2} \cos\!\left(\theta\right)^{2}-r^{11} M+\frac{1}{4} r^{12}+\frac{1}{4} a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{r t r}$ | $\frac{-2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-7 r M a^{10} \cos\!\left(\theta\right)^{4}+r M a^{10} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+6 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+12 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{2}+14 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}-2 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{3} M a^{8} \cos\!\left(\theta\right)^{2}-20 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}+2 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}+16 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-24 r^{3} M^{3} a^{6} \cos\!\left(\theta\right)^{2}-5 r^{4} M^{2} a^{6}+5 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2}-12 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+7 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+45 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+25 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}-r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{6}+3 r^{5} M a^{6}-4 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-13 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-19 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}+r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+8 r^{5} M^{3} a^{4}-8 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{2}+16 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-48 r^{5} M^{3} a^{4} \cos\!\left(\theta\right)^{2}-18 r^{6} M^{2} a^{4}+6 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+54 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+12 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+8 r^{7} M a^{4}-2 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-14 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-6 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+16 r^{7} M^{3} a^{2}-4 r^{7} M^{3} a^{2} \sin\!\left(\theta\right)^{2}-24 r^{7} M^{3} a^{2} \cos\!\left(\theta\right)^{2}-21 r^{8} M^{2} a^{2}+r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+21 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+7 r^{9} M a^{2}-5 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+8 r^{9} M^{3}-8 r^{10} M^{2}+2 r^{11} M-M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+M^{2} a^{10} \cos\!\left(\theta\right)^{4}-M^{2} a^{10} \cos\!\left(\theta\right)^{6}}{4 r M a^{12} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{12} \cos\!\left(\theta\right)^{6}-2 r M a^{12} \cos\!\left(\theta\right)^{8}-8 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{10} \cos\!\left(\theta\right)^{4}+8 r^{2} M^{2} a^{10} \cos\!\left(\theta\right)^{6}+4 r^{2} a^{12} \cos\!\left(\theta\right)^{6}+3 r^{2} a^{12} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{3} M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{10} \cos\!\left(\theta\right)^{4}-20 r^{3} M a^{10} \cos\!\left(\theta\right)^{6}-4 r^{3} M a^{10} \cos\!\left(\theta\right)^{8}+16 r^{3} M^{3} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{8} \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-40 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{2}+36 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+16 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{6}+6 r^{4} a^{10} \cos\!\left(\theta\right)^{4}+12 r^{4} a^{10} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+24 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{5} M a^{8} \cos\!\left(\theta\right)^{2}-48 r^{5} M a^{8} \cos\!\left(\theta\right)^{4}-28 r^{5} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{5} M a^{8} \cos\!\left(\theta\right)^{8}+32 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+16 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-16 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{6} \cos\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{6}-8 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2}-56 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-32 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+48 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+60 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} a^{8} \cos\!\left(\theta\right)^{2}+18 r^{6} a^{8} \cos\!\left(\theta\right)^{4}+12 r^{6} a^{8} \cos\!\left(\theta\right)^{6}+r^{6} a^{8} \cos\!\left(\theta\right)^{8}-4 r^{7} M a^{6}+4 r^{7} M a^{6} \sin\!\left(\theta\right)^{2}+24 r^{7} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{7} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-44 r^{7} M a^{6} \cos\!\left(\theta\right)^{2}-60 r^{7} M a^{6} \cos\!\left(\theta\right)^{4}-12 r^{7} M a^{6} \cos\!\left(\theta\right)^{6}-8 r^{7} M^{3} a^{4}+16 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{2}+32 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{4}-32 r^{7} M^{3} a^{4} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{4} \cos\!\left(\theta\right)^{4}+20 r^{8} M^{2} a^{4}-24 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-40 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+72 r^{8} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+28 r^{8} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+r^{8} a^{6}+12 r^{8} a^{6} \cos\!\left(\theta\right)^{2}+18 r^{8} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{8} a^{6} \cos\!\left(\theta\right)^{6}-14 r^{9} M a^{4}+8 r^{9} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{9} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-52 r^{9} M a^{4} \cos\!\left(\theta\right)^{2}-24 r^{9} M a^{4} \cos\!\left(\theta\right)^{4}-16 r^{9} M^{3} a^{2}+16 r^{9} M^{3} a^{2} \sin\!\left(\theta\right)^{2}-16 r^{9} M^{3} a^{2} \cos\!\left(\theta\right)^{2}+28 r^{10} M^{2} a^{2}-16 r^{10} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+32 r^{10} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+3 r^{10} a^{4}+12 r^{10} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{10} a^{4} \cos\!\left(\theta\right)^{4}-16 r^{11} M a^{2}+4 r^{11} M a^{2} \sin\!\left(\theta\right)^{2}-20 r^{11} M a^{2} \cos\!\left(\theta\right)^{2}-8 r^{11} M^{3}+12 r^{12} M^{2}+3 r^{12} a^{2}+4 r^{12} a^{2} \cos\!\left(\theta\right)^{2}-6 r^{13} M+r^{14}+a^{14} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{r t \theta}$ | $\frac{-r M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+r M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+\frac{3}{4} M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-\frac{3}{2} r^{2} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+\frac{3}{2} r^{2} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+r^{2} M^{3} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-4 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-\frac{9}{4} r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-3 r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+\frac{3}{4} r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-3 r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+3 r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+10 r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-4 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-\frac{9}{2} r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-\frac{3}{2} r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-3 r^{6} M^{3} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+6 r^{7} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-\frac{9}{4} r^{8} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-r M a^{10} \cos\!\left(\theta\right)^{6}-2 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+r^{2} a^{10} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{2} a^{10} \cos\!\left(\theta\right)^{8}+3 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-3 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+2 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+\frac{3}{2} r^{4} a^{8} \cos\!\left(\theta\right)^{4}+2 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{4} r^{4} a^{8} \cos\!\left(\theta\right)^{8}+3 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-6 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+r^{6} M^{2} a^{4}-2 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+r^{6} a^{6} \cos\!\left(\theta\right)^{2}+3 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+r^{6} a^{6} \cos\!\left(\theta\right)^{6}-r^{7} M a^{4}+r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+3 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-6 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-3 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{8} M^{2} a^{2}-2 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+2 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{4} r^{8} a^{4}+2 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+\frac{3}{2} r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-3 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} M^{2}+\frac{1}{2} r^{10} a^{2}+r^{10} a^{2} \cos\!\left(\theta\right)^{2}-r^{11} M+\frac{1}{4} r^{12}+\frac{1}{4} a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{r r \varphi}$ | $\frac{7 r M a^{11} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r M a^{11} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+2 r M a^{11} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-12 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-18 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-6 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+8 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-2 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+4 r^{3} M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+25 r^{3} M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+2 r^{3} M a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{3} M a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+24 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-16 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-8 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+5 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{2}-49 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-49 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-5 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{4}+16 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{7} \sin\!\left(\theta\right)^{2}+14 r^{5} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+29 r^{5} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{5} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+4 r^{5} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{5} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-8 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{2}+64 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{4}-32 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+18 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{2}-70 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-20 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-6 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{4}+16 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-9 r^{7} M a^{5} \sin\!\left(\theta\right)^{2}+16 r^{7} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+11 r^{7} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{7} M a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M^{3} a^{3} \sin\!\left(\theta\right)^{2}+24 r^{7} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+21 r^{8} M^{2} a^{3} \sin\!\left(\theta\right)^{2}-21 r^{8} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{8} M^{2} a^{3} \sin\!\left(\theta\right)^{4}-9 r^{9} M a^{3} \sin\!\left(\theta\right)^{2}+6 r^{9} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-12 r^{9} M^{3} a \sin\!\left(\theta\right)^{2}+12 r^{10} M^{2} a \sin\!\left(\theta\right)^{2}-3 r^{11} M a \sin\!\left(\theta\right)^{2}-M^{2} a^{11} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+M^{2} a^{11} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+M^{2} a^{11} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}}{4 r M a^{12} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{12} \cos\!\left(\theta\right)^{6}-2 r M a^{12} \cos\!\left(\theta\right)^{8}-8 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{10} \cos\!\left(\theta\right)^{4}+8 r^{2} M^{2} a^{10} \cos\!\left(\theta\right)^{6}+4 r^{2} a^{12} \cos\!\left(\theta\right)^{6}+3 r^{2} a^{12} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{3} M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{10} \cos\!\left(\theta\right)^{4}-20 r^{3} M a^{10} \cos\!\left(\theta\right)^{6}-4 r^{3} M a^{10} \cos\!\left(\theta\right)^{8}+16 r^{3} M^{3} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{8} \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-40 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{2}+36 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+16 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{6}+6 r^{4} a^{10} \cos\!\left(\theta\right)^{4}+12 r^{4} a^{10} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+24 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{5} M a^{8} \cos\!\left(\theta\right)^{2}-48 r^{5} M a^{8} \cos\!\left(\theta\right)^{4}-28 r^{5} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{5} M a^{8} \cos\!\left(\theta\right)^{8}+32 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+16 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-16 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{6} \cos\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{6}-8 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2}-56 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-32 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+48 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+60 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} a^{8} \cos\!\left(\theta\right)^{2}+18 r^{6} a^{8} \cos\!\left(\theta\right)^{4}+12 r^{6} a^{8} \cos\!\left(\theta\right)^{6}+r^{6} a^{8} \cos\!\left(\theta\right)^{8}-4 r^{7} M a^{6}+4 r^{7} M a^{6} \sin\!\left(\theta\right)^{2}+24 r^{7} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{7} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-44 r^{7} M a^{6} \cos\!\left(\theta\right)^{2}-60 r^{7} M a^{6} \cos\!\left(\theta\right)^{4}-12 r^{7} M a^{6} \cos\!\left(\theta\right)^{6}-8 r^{7} M^{3} a^{4}+16 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{2}+32 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{4}-32 r^{7} M^{3} a^{4} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{4} \cos\!\left(\theta\right)^{4}+20 r^{8} M^{2} a^{4}-24 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-40 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+72 r^{8} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+28 r^{8} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+r^{8} a^{6}+12 r^{8} a^{6} \cos\!\left(\theta\right)^{2}+18 r^{8} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{8} a^{6} \cos\!\left(\theta\right)^{6}-14 r^{9} M a^{4}+8 r^{9} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{9} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-52 r^{9} M a^{4} \cos\!\left(\theta\right)^{2}-24 r^{9} M a^{4} \cos\!\left(\theta\right)^{4}-16 r^{9} M^{3} a^{2}+16 r^{9} M^{3} a^{2} \sin\!\left(\theta\right)^{2}-16 r^{9} M^{3} a^{2} \cos\!\left(\theta\right)^{2}+28 r^{10} M^{2} a^{2}-16 r^{10} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+32 r^{10} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+3 r^{10} a^{4}+12 r^{10} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{10} a^{4} \cos\!\left(\theta\right)^{4}-16 r^{11} M a^{2}+4 r^{11} M a^{2} \sin\!\left(\theta\right)^{2}-20 r^{11} M a^{2} \cos\!\left(\theta\right)^{2}-8 r^{11} M^{3}+12 r^{12} M^{2}+3 r^{12} a^{2}+4 r^{12} a^{2} \cos\!\left(\theta\right)^{2}-6 r^{13} M+r^{14}+a^{14} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{r \theta \varphi}$ | $\frac{4 r M^{2} a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r M^{2} a^{9} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-2 M a^{11} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-3 M a^{11} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-2 r^{2} M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+6 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-2 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-4 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+4 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+16 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+2 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+8 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+9 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+20 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+12 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-4 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-12 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-4 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-16 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-4 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-40 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+20 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+2 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+16 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+10 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+22 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+14 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+12 r^{6} M^{3} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-16 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-16 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-20 r^{7} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+8 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+14 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+13 r^{8} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-12 r^{9} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)+6 r^{10} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{4 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{10} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+6 r^{4} a^{8} \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} M^{2} a^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+8 r^{8} M^{2} a^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{8} a^{4}+8 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-12 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+4 r^{10} M^{2}+2 r^{10} a^{2}+4 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-4 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{\theta t r}$ | $\frac{-2 r M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-3 r M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+\frac{3}{2} M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-3 r^{2} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+3 r^{2} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+4 r^{2} M^{3} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-4 r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+8 r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-2 r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-8 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-\frac{9}{2} r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-6 r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+\frac{3}{2} r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-8 r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+8 r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+21 r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+4 r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-6 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-9 r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-3 r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-12 r^{6} M^{3} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+15 r^{7} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-\frac{9}{2} r^{8} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-2 r M a^{10} \cos\!\left(\theta\right)^{6}-4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+2 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{6}+r^{2} a^{10} \cos\!\left(\theta\right)^{8}+6 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-6 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-4 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+3 r^{4} a^{8} \cos\!\left(\theta\right)^{4}+4 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{4} a^{8} \cos\!\left(\theta\right)^{8}+6 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+6 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-6 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-2 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+2 r^{6} M^{2} a^{4}-4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+2 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+2 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+2 r^{6} a^{6} \cos\!\left(\theta\right)^{2}+6 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+2 r^{6} a^{6} \cos\!\left(\theta\right)^{6}-2 r^{7} M a^{4}+2 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+6 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-6 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+4 r^{8} M^{2} a^{2}-4 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+4 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{2} r^{8} a^{4}+4 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+3 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-4 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-6 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+2 r^{10} M^{2}+r^{10} a^{2}+2 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+\frac{1}{2} r^{12}+\frac{1}{2} a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{\theta t \theta}$ | $\frac{8 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r M a^{10} \cos\!\left(\theta\right)^{4}+2 r M a^{10} \cos\!\left(\theta\right)^{6}-10 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-16 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+12 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-2 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{2}-6 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}-4 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{6}+6 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+16 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+7 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}+20 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{3} M^{3} a^{6} \cos\!\left(\theta\right)^{2}+8 r^{3} M^{3} a^{6} \cos\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{6}+2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2}-32 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4}+12 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-10 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}-20 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}-4 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{6}-r^{5} M a^{6}-2 r^{5} M a^{6} \sin\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+8 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}+11 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}+2 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}-4 r^{5} M^{3} a^{4}-4 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{2}+20 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+8 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{5} M^{3} a^{4} \cos\!\left(\theta\right)^{2}+8 r^{5} M^{3} a^{4} \cos\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{4}+8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-22 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}-18 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}-14 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}-3 r^{7} M a^{4}-4 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+6 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}+5 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}-8 r^{7} M^{3} a^{2}-4 r^{7} M^{3} a^{2} \sin\!\left(\theta\right)^{2}+12 r^{7} M^{3} a^{2} \cos\!\left(\theta\right)^{2}+10 r^{8} M^{2} a^{2}+6 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}-10 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}-3 r^{9} M a^{2}-2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}+2 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}-4 r^{9} M^{3}+4 r^{10} M^{2}-r^{11} M}{4 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{10} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+6 r^{4} a^{8} \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} M^{2} a^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+8 r^{8} M^{2} a^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{8} a^{4}+8 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-12 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+4 r^{10} M^{2}+2 r^{10} a^{2}+4 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-4 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{\theta r \varphi}$ | $\frac{2 r M^{2} a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+4 r M^{2} a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{9} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-M a^{11} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-3 M a^{11} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-r^{2} M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+6 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-8 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+8 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+16 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+4 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+9 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+16 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+16 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-16 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-2 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-8 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-2 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-44 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-4 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+16 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+8 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+5 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+20 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+10 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+24 r^{6} M^{3} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-8 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-8 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-28 r^{7} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+7 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+11 r^{8} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-6 r^{9} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)+3 r^{10} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{4 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{10} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+6 r^{4} a^{8} \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} M^{2} a^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+8 r^{8} M^{2} a^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{8} a^{4}+8 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-12 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+4 r^{10} M^{2}+2 r^{10} a^{2}+4 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-4 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{\theta \theta \varphi}$ | $\frac{-r M a^{11} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r M a^{11} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r M a^{11} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+2 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+18 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+16 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+10 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-12 r^{2} M^{2} a^{9} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-17 r^{3} M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-16 r^{3} M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-6 r^{3} M a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-16 r^{3} M a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-4 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-32 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-20 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+24 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+24 r^{3} M^{3} a^{7} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-2 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{2}+18 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+64 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+16 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-2 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{4}+24 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{6}-12 r^{4} M^{2} a^{7} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+r^{5} M a^{7} \sin\!\left(\theta\right)^{2}-4 r^{5} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-31 r^{5} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+2 r^{5} M a^{7} \sin\!\left(\theta\right)^{4}-12 r^{5} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-8 r^{5} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{2}-32 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-32 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{4}-4 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-8 r^{5} M^{3} a^{5} \sin\!\left(\theta\right)^{6}-12 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{2}+38 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+46 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{4}+14 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{5} \sin\!\left(\theta\right)^{6}+5 r^{7} M a^{5} \sin\!\left(\theta\right)^{2}-8 r^{7} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-15 r^{7} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{7} M a^{5} \sin\!\left(\theta\right)^{4}-6 r^{7} M a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+16 r^{7} M^{3} a^{3} \sin\!\left(\theta\right)^{2}-28 r^{7} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{7} M^{3} a^{3} \sin\!\left(\theta\right)^{4}-22 r^{8} M^{2} a^{3} \sin\!\left(\theta\right)^{2}+22 r^{8} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{8} M^{2} a^{3} \sin\!\left(\theta\right)^{4}+7 r^{9} M a^{3} \sin\!\left(\theta\right)^{2}-4 r^{9} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+2 r^{9} M a^{3} \sin\!\left(\theta\right)^{4}+12 r^{9} M^{3} a \sin\!\left(\theta\right)^{2}-12 r^{10} M^{2} a \sin\!\left(\theta\right)^{2}+3 r^{11} M a \sin\!\left(\theta\right)^{2}}{4 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{10} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+6 r^{4} a^{8} \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} M^{2} a^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+8 r^{8} M^{2} a^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{8} a^{4}+8 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-12 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+4 r^{10} M^{2}+2 r^{10} a^{2}+4 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-4 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{\varphi t \varphi}$ | $\frac{\frac{3}{2} r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{2} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+3 r^{2} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-6 r^{3} M^{3} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-2 r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-3 r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+3 r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-\frac{1}{2} r^{5} M a^{4} \sin\!\left(\theta\right)^{2}+2 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+\frac{3}{2} r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{5} M^{3} a^{2} \sin\!\left(\theta\right)^{4}+r^{6} M^{2} a^{2} \sin\!\left(\theta\right)^{2}-2 r^{6} M^{2} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-r^{6} M^{2} a^{2} \sin\!\left(\theta\right)^{4}-r^{7} M a^{2} \sin\!\left(\theta\right)^{2}+r^{7} M a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{8} M^{2} \sin\!\left(\theta\right)^{2}-\frac{1}{2} r^{9} M \sin\!\left(\theta\right)^{2}}{r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-r M a^{8} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{2} a^{8} \cos\!\left(\theta\right)^{8}+3 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{3} M a^{6} \cos\!\left(\theta\right)^{4}-r^{3} M a^{6} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{6} \cos\!\left(\theta\right)^{4}+2 r^{4} a^{6} \cos\!\left(\theta\right)^{6}+3 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{6} a^{4} \cos\!\left(\theta\right)^{2}+3 r^{6} a^{4} \cos\!\left(\theta\right)^{4}-r^{7} M a^{2}+r^{7} M a^{2} \sin\!\left(\theta\right)^{2}-3 r^{7} M a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{2} r^{8} a^{2}+2 r^{8} a^{2} \cos\!\left(\theta\right)^{2}-r^{9} M+\frac{1}{2} r^{10}+\frac{1}{2} a^{10} \cos\!\left(\theta\right)^{8}}$ |
| $R^{t}{}_{\varphi r \theta}$ | $\frac{-2 r M^{2} a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+M a^{11} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+r^{2} M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-2 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-4 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+4 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-4 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-4 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-2 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+4 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+2 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+8 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+2 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-4 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-8 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-8 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-5 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-2 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-4 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+12 r^{6} M^{3} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+8 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+8 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-8 r^{7} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-4 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-7 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-2 r^{8} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+6 r^{9} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)-3 r^{10} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{4 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{10} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+6 r^{4} a^{8} \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} M^{2} a^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+8 r^{8} M^{2} a^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{8} a^{4}+8 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-12 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+4 r^{10} M^{2}+2 r^{10} a^{2}+4 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-4 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{r}{}_{t t r}$ | $\frac{-2 r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-7 r M a^{8} \cos\!\left(\theta\right)^{4}+r M a^{8} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{2} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{2} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+12 r^{2} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+13 r^{2} M^{2} a^{6} \cos\!\left(\theta\right)^{4}-r^{2} M^{2} a^{6} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{3} M a^{6} \cos\!\left(\theta\right)^{2}-13 r^{3} M a^{6} \cos\!\left(\theta\right)^{4}+r^{3} M a^{6} \cos\!\left(\theta\right)^{6}+24 r^{3} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{3} M^{3} a^{4} \cos\!\left(\theta\right)^{2}-5 r^{4} M^{2} a^{4}+5 r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+33 r^{4} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+12 r^{4} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+3 r^{5} M a^{4}-2 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-9 r^{5} M a^{4} \cos\!\left(\theta\right)^{2}-6 r^{5} M a^{4} \cos\!\left(\theta\right)^{4}+8 r^{5} M^{3} a^{2}-8 r^{5} M^{3} a^{2} \sin\!\left(\theta\right)^{2}-24 r^{5} M^{3} a^{2} \cos\!\left(\theta\right)^{2}-13 r^{6} M^{2} a^{2}+3 r^{6} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+21 r^{6} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+5 r^{7} M a^{2}-5 r^{7} M a^{2} \cos\!\left(\theta\right)^{2}+8 r^{7} M^{3}-8 r^{8} M^{2}+2 r^{9} M-M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+M^{2} a^{8} \cos\!\left(\theta\right)^{4}-M^{2} a^{8} \cos\!\left(\theta\right)^{6}}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{r}{}_{t t \theta}$ | $\frac{-8 r M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-16 r M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+8 r M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+6 M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-12 r^{2} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+12 r^{2} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+16 r^{2} M^{3} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+8 r^{2} M^{3} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-16 r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+32 r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+8 r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-32 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+8 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-18 r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-24 r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+6 r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-64 r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+16 r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+64 r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+104 r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+16 r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-32 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-36 r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-12 r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-72 r^{6} M^{3} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+72 r^{7} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-18 r^{8} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{4 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-4 r M a^{10} \cos\!\left(\theta\right)^{8}+10 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{10}+16 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-16 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-4 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+20 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+24 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-24 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-16 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+20 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+20 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+16 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-16 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-24 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+20 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-4 r^{9} M a^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-16 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+2 r^{10} a^{2}+10 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-4 r^{11} M+2 r^{12}+2 a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{r}{}_{t r \varphi}$ | $\frac{7 r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+2 r M a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-12 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-17 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+8 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+3 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+4 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+18 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+2 r^{3} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{3} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+24 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+5 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2}-37 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-18 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-5 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4}+12 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{5} \sin\!\left(\theta\right)^{2}+10 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+11 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{5} M a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-8 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2}+24 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+8 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{4}+13 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2}-23 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-3 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{4}-6 r^{7} M a^{3} \sin\!\left(\theta\right)^{2}+6 r^{7} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a \sin\!\left(\theta\right)^{2}+10 r^{8} M^{2} a \sin\!\left(\theta\right)^{2}-3 r^{9} M a \sin\!\left(\theta\right)^{2}-M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+M^{2} a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{r}{}_{t \theta \varphi}$ | $\frac{4 r M^{2} a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+4 r M^{2} a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+4 r M^{2} a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{9} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-2 M a^{11} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-3 M a^{11} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-2 r^{2} M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+6 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-8 r^{2} M^{3} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-8 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+4 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+8 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-16 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+16 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+6 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+6 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+9 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+16 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+8 r^{4} M^{3} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+32 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-16 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-32 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-8 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-28 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-12 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-44 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-12 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+16 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+4 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+18 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+6 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+20 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+10 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+16 r^{6} M^{3} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+24 r^{6} M^{3} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+20 r^{6} M^{3} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-28 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-36 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-32 r^{7} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+10 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+12 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+11 r^{8} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+24 r^{8} M^{3} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)-24 r^{9} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)+6 r^{10} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{r}{}_{\theta t \varphi}$ | $\frac{2 r M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+2 r M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-2 r M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r^{2} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r^{2} M^{3} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+4 r^{2} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-2 r^{3} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-2 r^{3} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-6 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+r^{4} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+4 r^{4} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+r^{4} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r^{4} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+2 r^{4} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+4 r^{4} M^{3} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+4 r^{4} M^{3} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-4 r^{4} M^{3} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-10 r^{5} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-10 r^{5} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-2 r^{5} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{6} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+4 r^{6} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+2 r^{6} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+12 r^{6} M^{3} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)-12 r^{7} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)+3 r^{8} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{2 r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-2 r M a^{8} \cos\!\left(\theta\right)^{6}+4 r^{2} a^{8} \cos\!\left(\theta\right)^{6}+r^{2} a^{8} \cos\!\left(\theta\right)^{8}+6 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-6 r^{3} M a^{6} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{6} \cos\!\left(\theta\right)^{6}+6 r^{4} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{4} a^{6} \cos\!\left(\theta\right)^{6}+6 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-6 r^{5} M a^{4} \cos\!\left(\theta\right)^{2}-6 r^{5} M a^{4} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{6} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{7} M a^{2}+2 r^{7} M a^{2} \sin\!\left(\theta\right)^{2}-6 r^{7} M a^{2} \cos\!\left(\theta\right)^{2}+r^{8} a^{2}+4 r^{8} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{9} M+r^{10}+a^{10} \cos\!\left(\theta\right)^{8}}$ |
| $R^{r}{}_{\theta r \theta}$ | $\frac{3 r M a^{2} \cos\!\left(\theta\right)^{2}+r^{2} a^{2}-r^{2} a^{2} \sin\!\left(\theta\right)^{2}-r^{2} a^{2} \cos\!\left(\theta\right)^{2}-r^{3} M+a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-a^{4} \cos\!\left(\theta\right)^{2}+a^{4} \cos\!\left(\theta\right)^{4}}{2 r^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{4}+a^{4} \cos\!\left(\theta\right)^{4}}$ |
| $R^{r}{}_{\varphi t r}$ | $\frac{-7 r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-2 r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-2 r M a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+12 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+17 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-3 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-4 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-4 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-18 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-2 r^{3} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-24 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+24 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-5 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2}+37 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+18 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+5 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4}-12 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+3 r^{5} M a^{5} \sin\!\left(\theta\right)^{2}-10 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-11 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-2 r^{5} M a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2}-24 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{4}-13 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2}+23 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{4}+6 r^{7} M a^{3} \sin\!\left(\theta\right)^{2}-6 r^{7} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+8 r^{7} M^{3} a \sin\!\left(\theta\right)^{2}-10 r^{8} M^{2} a \sin\!\left(\theta\right)^{2}+3 r^{9} M a \sin\!\left(\theta\right)^{2}+M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-M^{2} a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{r}{}_{\varphi t \theta}$ | $\frac{-2 r M^{2} a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-2 r M^{2} a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-4 r M^{2} a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-6 r M^{2} a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{9} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+M a^{11} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+3 M a^{11} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-r^{2} M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r^{2} M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-6 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+6 r^{2} M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+4 r^{2} M^{3} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+8 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-8 r^{2} M^{3} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-2 r^{3} M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-2 r^{3} M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+16 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-6 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-16 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+4 r^{3} M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-5 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-2 r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+r^{4} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-9 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-12 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+3 r^{4} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-8 r^{4} M^{3} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+4 r^{4} M^{3} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-32 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+16 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+32 r^{4} M^{3} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+6 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+18 r^{5} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+46 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-16 r^{5} M^{2} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-3 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-10 r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{6} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-18 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-6 r^{6} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-12 r^{6} M^{3} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-8 r^{6} M^{3} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-24 r^{6} M^{3} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+18 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+14 r^{7} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+30 r^{7} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-6 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-5 r^{8} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-9 r^{8} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-12 r^{8} M^{3} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)+12 r^{9} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)-3 r^{10} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{r}{}_{\varphi r \varphi}$ | $\frac{3 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}+7 r M a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+5 r M a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{6}+2 r M a^{10} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{4}-6 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-21 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+3 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{6}+8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+7 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{8} \cos\!\left(\theta\right)^{2}+10 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+23 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+3 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{6}+2 r^{3} M a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+2 r^{3} M a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{4}+24 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-24 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-14 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+5 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4}-41 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-10 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-5 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{6}+16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-3 r^{5} M a^{6} \sin\!\left(\theta\right)^{4}+11 r^{5} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+10 r^{5} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+2 r^{5} M a^{6} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-8 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{4}+24 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{6}-10 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+13 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}-15 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-3 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{6}+6 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-7 r^{7} M a^{4} \sin\!\left(\theta\right)^{4}+r^{7} M a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{2} \sin\!\left(\theta\right)^{4}-2 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+2 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+14 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{4}+r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-6 r^{9} M a^{2} \sin\!\left(\theta\right)^{4}+2 r^{10} M^{2} \sin\!\left(\theta\right)^{2}-r^{11} M \sin\!\left(\theta\right)^{2}-M^{2} a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+M^{2} a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{6}+M^{2} a^{10} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{4}}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{r}{}_{\varphi \theta \varphi}$ | $\frac{6 r M^{2} a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+6 r M^{2} a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}+4 r M^{2} a^{10} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+2 r M^{2} a^{10} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{10} \sin\!\left(\theta\right)^{7} \cos\!\left(\theta\right)^{3}+r a^{12} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-r a^{12} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{9}-r a^{12} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}-3 M a^{12} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}-3 M a^{12} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}-2 r^{2} M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r^{2} M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{9}+5 r^{2} M a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-4 r^{2} M a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}+6 r^{2} M a^{10} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-6 r^{2} M a^{10} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}-12 r^{2} M^{3} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-8 r^{2} M^{3} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+8 r^{2} M^{3} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}+8 r^{2} M^{3} a^{8} \sin\!\left(\theta\right)^{7} \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-4 r^{3} M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-8 r^{3} M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-2 r^{3} M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+6 r^{3} M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}-16 r^{3} M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+4 r^{3} M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+2 r^{3} M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}+16 r^{3} M^{2} a^{8} \sin\!\left(\theta\right)^{7} \cos\!\left(\theta\right)-4 r^{3} M^{2} a^{8} \sin\!\left(\theta\right)^{7} \cos\!\left(\theta\right)^{3}+3 r^{3} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-r^{3} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-2 r^{3} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{9}-3 r^{3} a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-2 r^{3} a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}-4 r^{4} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-6 r^{4} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+8 r^{4} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+2 r^{4} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{9}+19 r^{4} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+16 r^{4} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-r^{4} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}+9 r^{4} M a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+12 r^{4} M a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-3 r^{4} M a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}+16 r^{4} M^{3} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r^{4} M^{3} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+32 r^{4} M^{3} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-24 r^{4} M^{3} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-32 r^{4} M^{3} a^{6} \sin\!\left(\theta\right)^{7} \cos\!\left(\theta\right)+8 r^{5} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-4 r^{5} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-4 r^{5} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-14 r^{5} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-62 r^{5} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-8 r^{5} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-38 r^{5} M^{2} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+16 r^{5} M^{2} a^{6} \sin\!\left(\theta\right)^{7} \cos\!\left(\theta\right)+3 r^{5} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+3 r^{5} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-5 r^{5} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-r^{5} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{9}-3 r^{5} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-6 r^{5} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-r^{5} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}-2 r^{6} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-12 r^{6} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+6 r^{6} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+8 r^{6} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+11 r^{6} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+44 r^{6} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+11 r^{6} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+18 r^{6} M a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+6 r^{6} M a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+28 r^{6} M^{3} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+32 r^{6} M^{3} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+8 r^{6} M^{3} a^{4} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+4 r^{7} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+4 r^{7} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-8 r^{7} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-54 r^{7} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-54 r^{7} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-22 r^{7} M^{2} a^{4} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+r^{7} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+5 r^{7} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-3 r^{7} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-3 r^{7} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-r^{7} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-6 r^{7} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-3 r^{7} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-6 r^{8} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-4 r^{8} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+10 r^{8} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+24 r^{8} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+25 r^{8} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+9 r^{8} M a^{4} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+36 r^{8} M^{3} a^{2} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+4 r^{9} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-4 r^{9} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-40 r^{9} M^{2} a^{2} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+2 r^{9} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+r^{9} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-3 r^{9} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-2 r^{9} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-3 r^{9} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r^{10} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+4 r^{10} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+13 r^{10} M a^{2} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+r^{11} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-r^{11} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{11} a^{2} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\theta}{}_{t t r}$ | $\frac{-4 r M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-2 r M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+3 M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-6 r^{2} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+3 r^{2} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+16 r^{3} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-4 r^{3} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-9 r^{4} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-6 r^{4} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+18 r^{5} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-9 r^{6} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\theta}{}_{t t \theta}$ | $\frac{8 r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r M a^{8} \cos\!\left(\theta\right)^{4}+2 r M a^{8} \cos\!\left(\theta\right)^{6}-10 r^{2} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{2} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+12 r^{2} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-2 r^{2} M^{2} a^{6} \cos\!\left(\theta\right)^{2}-6 r^{2} M^{2} a^{6} \cos\!\left(\theta\right)^{4}-4 r^{2} M^{2} a^{6} \cos\!\left(\theta\right)^{6}+6 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+8 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+6 r^{3} M a^{6} \cos\!\left(\theta\right)^{4}+2 r^{3} M a^{6} \cos\!\left(\theta\right)^{6}-4 r^{3} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{3} M^{3} a^{4} \cos\!\left(\theta\right)^{2}+8 r^{3} M^{3} a^{4} \cos\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{4}+2 r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-14 r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{4} M^{2} a^{4} \sin\!\left(\theta\right)^{4}-8 r^{4} M^{2} a^{4} \cos\!\left(\theta\right)^{2}-14 r^{4} M^{2} a^{4} \cos\!\left(\theta\right)^{4}-r^{5} M a^{4}-2 r^{5} M a^{4} \sin\!\left(\theta\right)^{2}+6 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+2 r^{5} M a^{4} \cos\!\left(\theta\right)^{2}+5 r^{5} M a^{4} \cos\!\left(\theta\right)^{4}-4 r^{5} M^{3} a^{2}+4 r^{5} M^{3} a^{2} \sin\!\left(\theta\right)^{2}+12 r^{5} M^{3} a^{2} \cos\!\left(\theta\right)^{2}+6 r^{6} M^{2} a^{2}+2 r^{6} M^{2} a^{2} \sin\!\left(\theta\right)^{2}-10 r^{6} M^{2} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{7} M a^{2}-2 r^{7} M a^{2} \sin\!\left(\theta\right)^{2}+2 r^{7} M a^{2} \cos\!\left(\theta\right)^{2}-4 r^{7} M^{3}+4 r^{8} M^{2}-r^{9} M}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\theta}{}_{t r \varphi}$ | $\frac{2 r M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-3 M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+r^{2} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-r^{2} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+6 r^{2} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-3 r^{2} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-4 r^{3} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+2 r^{3} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-16 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+8 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+16 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+5 r^{4} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+r^{4} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+9 r^{4} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+6 r^{4} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-6 r^{5} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-4 r^{5} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-12 r^{5} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+3 r^{6} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+5 r^{6} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+9 r^{6} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-6 r^{7} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)+3 r^{8} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\theta}{}_{t \theta \varphi}$ | $\frac{-r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r M a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+2 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+18 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+10 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-8 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-12 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-16 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-6 r^{3} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-8 r^{3} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-4 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-2 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2}+16 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+26 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-2 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4}+6 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{6}+r^{5} M a^{5} \sin\!\left(\theta\right)^{2}-4 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-15 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{5} M a^{5} \sin\!\left(\theta\right)^{4}-6 r^{5} M a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2}-12 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{4}-10 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2}+18 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+2 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{4}+4 r^{7} M a^{3} \sin\!\left(\theta\right)^{2}-4 r^{7} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+2 r^{7} M a^{3} \sin\!\left(\theta\right)^{4}+4 r^{7} M^{3} a \sin\!\left(\theta\right)^{2}-8 r^{8} M^{2} a \sin\!\left(\theta\right)^{2}+3 r^{9} M a \sin\!\left(\theta\right)^{2}}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\theta}{}_{r t \varphi}$ | $\frac{-r M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+r M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+\frac{1}{2} M a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-\frac{1}{2} r^{2} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-r^{2} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+r^{3} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+r^{3} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{3} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-\frac{1}{2} r^{4} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-2 r^{4} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{4} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+3 r^{5} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)-\frac{3}{2} r^{6} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-r M a^{8} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{2} a^{8} \cos\!\left(\theta\right)^{8}+3 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{3} M a^{6} \cos\!\left(\theta\right)^{4}-r^{3} M a^{6} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{6} \cos\!\left(\theta\right)^{4}+2 r^{4} a^{6} \cos\!\left(\theta\right)^{6}+3 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{6} a^{4} \cos\!\left(\theta\right)^{2}+3 r^{6} a^{4} \cos\!\left(\theta\right)^{4}-r^{7} M a^{2}+r^{7} M a^{2} \sin\!\left(\theta\right)^{2}-3 r^{7} M a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{2} r^{8} a^{2}+2 r^{8} a^{2} \cos\!\left(\theta\right)^{2}-r^{9} M+\frac{1}{2} r^{10}+\frac{1}{2} a^{10} \cos\!\left(\theta\right)^{8}}$ |
| $R^{\theta}{}_{r r \theta}$ | $\frac{-3 r M a^{2} \cos\!\left(\theta\right)^{2}-r^{2} a^{2}+r^{2} a^{2} \sin\!\left(\theta\right)^{2}+r^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{3} M-a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+a^{4} \cos\!\left(\theta\right)^{2}-a^{4} \cos\!\left(\theta\right)^{4}}{-2 r M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{2} a^{4} \cos\!\left(\theta\right)^{4}-4 r^{3} M a^{2} \cos\!\left(\theta\right)^{2}+r^{4} a^{2}+2 r^{4} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M+r^{6}+a^{6} \cos\!\left(\theta\right)^{4}}$ |
| $R^{\theta}{}_{\varphi t r}$ | $\frac{-4 r M^{2} a^{7} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+2 r M^{2} a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{7} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+2 M a^{9} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+3 M a^{9} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-6 r^{2} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+r^{2} M a^{7} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+4 r^{3} M^{2} a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+16 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-8 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-6 r^{4} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-6 r^{4} M a^{5} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-9 r^{4} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-10 r^{4} M a^{5} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+8 r^{5} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+12 r^{5} M^{2} a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+10 r^{5} M^{2} a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-4 r^{6} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-12 r^{6} M a^{3} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-11 r^{6} M a^{3} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+12 r^{7} M^{2} a \sin\!\left(\theta\right) \cos\!\left(\theta\right)-6 r^{8} M a \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\theta}{}_{\varphi t \theta}$ | $\frac{r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+8 r M a^{9} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-2 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-18 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-10 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+12 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+16 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+6 r^{3} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{3} M a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+8 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-4 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2}-16 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-26 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4}-6 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-4 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{6}-r^{5} M a^{5} \sin\!\left(\theta\right)^{2}+4 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+15 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-2 r^{5} M a^{5} \sin\!\left(\theta\right)^{4}+6 r^{5} M a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-4 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2}+12 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{4}+10 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2}-18 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{4}-4 r^{7} M a^{3} \sin\!\left(\theta\right)^{2}+4 r^{7} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{7} M a^{3} \sin\!\left(\theta\right)^{4}-4 r^{7} M^{3} a \sin\!\left(\theta\right)^{2}+8 r^{8} M^{2} a \sin\!\left(\theta\right)^{2}-3 r^{9} M a \sin\!\left(\theta\right)^{2}}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\theta}{}_{\varphi r \varphi}$ | $\frac{6 r M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-4 r M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{8} \sin\!\left(\theta\right)^{7} \cos\!\left(\theta\right)^{3}+r a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-r a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{9}-r a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}-3 M a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}-3 M a^{10} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}-2 r^{2} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r^{2} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+5 r^{2} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-3 r^{2} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}+6 r^{2} M a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-3 r^{2} M a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{5}-8 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+2 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-16 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+12 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}+16 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{7} \cos\!\left(\theta\right)+3 r^{3} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-2 r^{3} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-r^{3} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{9}-3 r^{3} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-r^{3} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{7}-4 r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+2 r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+2 r^{4} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}+19 r^{4} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+5 r^{4} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}+9 r^{4} M a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+6 r^{4} M a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{3}-14 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-16 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-4 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)+3 r^{5} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-3 r^{5} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{7}-3 r^{5} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-3 r^{5} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{5}-2 r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-2 r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+4 r^{6} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+11 r^{6} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+19 r^{6} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+9 r^{6} M a^{4} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)-18 r^{7} M^{2} a^{2} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+r^{7} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+2 r^{7} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-3 r^{7} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}-r^{7} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-3 r^{7} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-2 r^{8} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+2 r^{8} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+11 r^{8} M a^{2} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+r^{9} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-r^{9} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{9} a^{2} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\theta}{}_{\varphi \theta \varphi}$ | $\frac{-6 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-r M a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-14 r M a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{6}-8 r M a^{10} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{4}+12 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+2 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+30 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{6}+10 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-20 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{4}-12 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{8} \cos\!\left(\theta\right)^{2}-16 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-6 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-26 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-14 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{6}-6 r^{3} M a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-8 r^{3} M a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{4}-4 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-8 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+20 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+12 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4}+24 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+18 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{6}-2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{8}-12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-16 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+r^{5} M a^{6} \sin\!\left(\theta\right)^{4}-6 r^{5} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-25 r^{5} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+2 r^{5} M a^{6} \sin\!\left(\theta\right)^{6}-6 r^{5} M a^{6} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+4 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{4}-12 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-4 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{6}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+20 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-14 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+22 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+6 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{6}-12 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+6 r^{7} M a^{4} \sin\!\left(\theta\right)^{4}-6 r^{7} M a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{7} M a^{4} \sin\!\left(\theta\right)^{6}+4 r^{7} M^{3} a^{2} \sin\!\left(\theta\right)^{4}-4 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+4 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{4}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}+5 r^{9} M a^{2} \sin\!\left(\theta\right)^{4}-4 r^{10} M^{2} \sin\!\left(\theta\right)^{2}+2 r^{11} M \sin\!\left(\theta\right)^{2}}{2 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{8}-2 r M a^{10} \cos\!\left(\theta\right)^{8}+5 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+r^{2} a^{10} \cos\!\left(\theta\right)^{10}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+10 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+5 r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-8 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+10 r^{6} a^{6} \cos\!\left(\theta\right)^{6}+8 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+5 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+10 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+2 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} a^{2}+5 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{10}}$ |
| $R^{\varphi}{}_{t t \varphi}$ | $\frac{\frac{3}{2} r M a^{6} \cos\!\left(\theta\right)^{4}-3 r^{2} M^{2} a^{4} \cos\!\left(\theta\right)^{2}-3 r^{2} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+r^{3} M a^{4} \cos\!\left(\theta\right)^{2}+\frac{3}{2} r^{3} M a^{4} \cos\!\left(\theta\right)^{4}+6 r^{3} M^{3} a^{2} \cos\!\left(\theta\right)^{2}+r^{4} M^{2} a^{2}-5 r^{4} M^{2} a^{2} \cos\!\left(\theta\right)^{2}-\frac{1}{2} r^{5} M a^{2}+r^{5} M a^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M^{3}+2 r^{6} M^{2}-\frac{1}{2} r^{7} M}{r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-r M a^{8} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{2} a^{8} \cos\!\left(\theta\right)^{8}+3 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{3} M a^{6} \cos\!\left(\theta\right)^{4}-r^{3} M a^{6} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{6} \cos\!\left(\theta\right)^{4}+2 r^{4} a^{6} \cos\!\left(\theta\right)^{6}+3 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{6} a^{4} \cos\!\left(\theta\right)^{2}+3 r^{6} a^{4} \cos\!\left(\theta\right)^{4}-r^{7} M a^{2}+r^{7} M a^{2} \sin\!\left(\theta\right)^{2}-3 r^{7} M a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{2} r^{8} a^{2}+2 r^{8} a^{2} \cos\!\left(\theta\right)^{2}-r^{9} M+\frac{1}{2} r^{10}+\frac{1}{2} a^{10} \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{t r \theta}$ | $\frac{2 r M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{7} \cos\!\left(\theta\right)^{5}+M a^{9} \cos\!\left(\theta\right)^{7}-2 r^{2} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+r^{2} M a^{7} \cos\!\left(\theta\right)^{5}-r^{2} M a^{7} \cos\!\left(\theta\right)^{7}-4 r^{2} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+4 r^{2} M^{3} a^{5} \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{5} \cos\!\left(\theta\right)^{5}-4 r^{4} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-r^{4} M a^{5} \cos\!\left(\theta\right)^{3}-5 r^{4} M a^{5} \cos\!\left(\theta\right)^{5}-4 r^{4} M^{3} a^{3} \cos\!\left(\theta\right)+4 r^{4} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-4 r^{4} M^{3} a^{3} \cos\!\left(\theta\right)^{3}+4 r^{5} M^{2} a^{3} \cos\!\left(\theta\right)+2 r^{5} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+16 r^{5} M^{2} a^{3} \cos\!\left(\theta\right)^{3}-r^{6} M a^{3} \cos\!\left(\theta\right)-2 r^{6} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-7 r^{6} M a^{3} \cos\!\left(\theta\right)^{3}-12 r^{6} M^{3} a \cos\!\left(\theta\right)+12 r^{7} M^{2} a \cos\!\left(\theta\right)-3 r^{8} M a \cos\!\left(\theta\right)}{-4 r M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+4 r M a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{2}+6 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)+16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{5}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4} \sin\!\left(\theta\right)-24 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{3}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{3}+r^{8} a^{4} \sin\!\left(\theta\right)+8 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2} \sin\!\left(\theta\right)-12 r^{9} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{3}+4 r^{10} M^{2} \sin\!\left(\theta\right)+2 r^{10} a^{2} \sin\!\left(\theta\right)+4 r^{10} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-4 r^{11} M \sin\!\left(\theta\right)+r^{12} \sin\!\left(\theta\right)+a^{12} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{r t r}$ | $\frac{2 r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+7 r M a^{9} \cos\!\left(\theta\right)^{4}+2 r M a^{9} \cos\!\left(\theta\right)^{6}+8 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-12 r^{2} M^{2} a^{7} \cos\!\left(\theta\right)^{2}-23 r^{2} M^{2} a^{7} \cos\!\left(\theta\right)^{4}-r^{2} M^{2} a^{7} \cos\!\left(\theta\right)^{6}+2 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+2 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{7} \cos\!\left(\theta\right)^{2}+18 r^{3} M a^{7} \cos\!\left(\theta\right)^{4}+2 r^{3} M a^{7} \cos\!\left(\theta\right)^{6}-32 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+32 r^{3} M^{3} a^{5} \cos\!\left(\theta\right)^{2}+4 r^{3} M^{3} a^{5} \cos\!\left(\theta\right)^{4}+5 r^{4} M^{2} a^{5}-5 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2}+12 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-41 r^{4} M^{2} a^{5} \cos\!\left(\theta\right)^{2}-24 r^{4} M^{2} a^{5} \cos\!\left(\theta\right)^{4}-3 r^{5} M a^{5}+2 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+10 r^{5} M a^{5} \cos\!\left(\theta\right)^{2}+11 r^{5} M a^{5} \cos\!\left(\theta\right)^{4}-8 r^{5} M^{3} a^{3}+8 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2}+32 r^{5} M^{3} a^{3} \cos\!\left(\theta\right)^{2}+15 r^{6} M^{2} a^{3}-3 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2}-27 r^{6} M^{2} a^{3} \cos\!\left(\theta\right)^{2}-6 r^{7} M a^{3}+6 r^{7} M a^{3} \cos\!\left(\theta\right)^{2}-12 r^{7} M^{3} a+12 r^{8} M^{2} a-3 r^{9} M a+M^{2} a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-M^{2} a^{9} \cos\!\left(\theta\right)^{4}+M^{2} a^{9} \cos\!\left(\theta\right)^{6}}{4 r M a^{12} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{12} \cos\!\left(\theta\right)^{6}-2 r M a^{12} \cos\!\left(\theta\right)^{8}-8 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{10} \cos\!\left(\theta\right)^{4}+8 r^{2} M^{2} a^{10} \cos\!\left(\theta\right)^{6}+4 r^{2} a^{12} \cos\!\left(\theta\right)^{6}+3 r^{2} a^{12} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{3} M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{10} \cos\!\left(\theta\right)^{4}-20 r^{3} M a^{10} \cos\!\left(\theta\right)^{6}-4 r^{3} M a^{10} \cos\!\left(\theta\right)^{8}+16 r^{3} M^{3} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{8} \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-40 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{2}+36 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+16 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{6}+6 r^{4} a^{10} \cos\!\left(\theta\right)^{4}+12 r^{4} a^{10} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+24 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{5} M a^{8} \cos\!\left(\theta\right)^{2}-48 r^{5} M a^{8} \cos\!\left(\theta\right)^{4}-28 r^{5} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{5} M a^{8} \cos\!\left(\theta\right)^{8}+32 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+16 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-16 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{6} \cos\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{6}-8 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2}-56 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-32 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+48 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+60 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} a^{8} \cos\!\left(\theta\right)^{2}+18 r^{6} a^{8} \cos\!\left(\theta\right)^{4}+12 r^{6} a^{8} \cos\!\left(\theta\right)^{6}+r^{6} a^{8} \cos\!\left(\theta\right)^{8}-4 r^{7} M a^{6}+4 r^{7} M a^{6} \sin\!\left(\theta\right)^{2}+24 r^{7} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{7} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-44 r^{7} M a^{6} \cos\!\left(\theta\right)^{2}-60 r^{7} M a^{6} \cos\!\left(\theta\right)^{4}-12 r^{7} M a^{6} \cos\!\left(\theta\right)^{6}-8 r^{7} M^{3} a^{4}+16 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{2}+32 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{4}-32 r^{7} M^{3} a^{4} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{4} \cos\!\left(\theta\right)^{4}+20 r^{8} M^{2} a^{4}-24 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-40 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+72 r^{8} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+28 r^{8} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+r^{8} a^{6}+12 r^{8} a^{6} \cos\!\left(\theta\right)^{2}+18 r^{8} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{8} a^{6} \cos\!\left(\theta\right)^{6}-14 r^{9} M a^{4}+8 r^{9} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{9} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-52 r^{9} M a^{4} \cos\!\left(\theta\right)^{2}-24 r^{9} M a^{4} \cos\!\left(\theta\right)^{4}-16 r^{9} M^{3} a^{2}+16 r^{9} M^{3} a^{2} \sin\!\left(\theta\right)^{2}-16 r^{9} M^{3} a^{2} \cos\!\left(\theta\right)^{2}+28 r^{10} M^{2} a^{2}-16 r^{10} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+32 r^{10} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+3 r^{10} a^{4}+12 r^{10} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{10} a^{4} \cos\!\left(\theta\right)^{4}-16 r^{11} M a^{2}+4 r^{11} M a^{2} \sin\!\left(\theta\right)^{2}-20 r^{11} M a^{2} \cos\!\left(\theta\right)^{2}-8 r^{11} M^{3}+12 r^{12} M^{2}+3 r^{12} a^{2}+4 r^{12} a^{2} \cos\!\left(\theta\right)^{2}-6 r^{13} M+r^{14}+a^{14} \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{r t \theta}$ | $\frac{4 r M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-4 r M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+4 r M^{2} a^{7} \cos\!\left(\theta\right)^{5}-3 M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-M a^{9} \cos\!\left(\theta\right)^{7}+6 r^{2} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-3 r^{2} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+r^{2} M a^{7} \cos\!\left(\theta\right)^{5}-r^{2} M a^{7} \cos\!\left(\theta\right)^{7}+4 r^{2} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-4 r^{2} M^{3} a^{5} \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+8 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+16 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-8 r^{3} M^{2} a^{5} \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{5} \cos\!\left(\theta\right)^{5}+9 r^{4} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+6 r^{4} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+5 r^{4} M a^{5} \cos\!\left(\theta\right)^{3}+r^{4} M a^{5} \cos\!\left(\theta\right)^{5}+12 r^{4} M^{3} a^{3} \cos\!\left(\theta\right)-12 r^{4} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-4 r^{4} M^{3} a^{3} \cos\!\left(\theta\right)^{3}-12 r^{5} M^{2} a^{3} \cos\!\left(\theta\right)-12 r^{5} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-8 r^{5} M^{2} a^{3} \cos\!\left(\theta\right)^{3}+3 r^{6} M a^{3} \cos\!\left(\theta\right)+9 r^{6} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+5 r^{6} M a^{3} \cos\!\left(\theta\right)^{3}+12 r^{6} M^{3} a \cos\!\left(\theta\right)-12 r^{7} M^{2} a \cos\!\left(\theta\right)+3 r^{8} M a \cos\!\left(\theta\right)}{-4 r M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+4 r M a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{2}+6 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)+16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{5}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4} \sin\!\left(\theta\right)-24 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{3}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{3}+r^{8} a^{4} \sin\!\left(\theta\right)+8 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2} \sin\!\left(\theta\right)-12 r^{9} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{3}+4 r^{10} M^{2} \sin\!\left(\theta\right)+2 r^{10} a^{2} \sin\!\left(\theta\right)+4 r^{10} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-4 r^{11} M \sin\!\left(\theta\right)+r^{12} \sin\!\left(\theta\right)+a^{12} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{r r \varphi}$ | $\frac{-7 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-5 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-2 r M a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-3 r M a^{10} \cos\!\left(\theta\right)^{8}+12 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+27 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-7 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}+12 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{6}-4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-23 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-2 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-2 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-10 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-r^{3} M a^{8} \cos\!\left(\theta\right)^{8}-32 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+8 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+32 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-12 r^{3} M^{3} a^{6} \cos\!\left(\theta\right)^{4}-5 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2}+45 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+5 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+28 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{6}+3 r^{5} M a^{6} \sin\!\left(\theta\right)^{2}-11 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-10 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-2 r^{5} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-2 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+8 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{4}-16 r^{5} M^{3} a^{4} \cos\!\left(\theta\right)^{2}-4 r^{5} M^{3} a^{4} \cos\!\left(\theta\right)^{4}-15 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}+11 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+20 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+7 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}-r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-6 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-4 r^{7} M^{3} a^{2}+16 r^{7} M^{3} a^{2} \sin\!\left(\theta\right)^{2}+4 r^{8} M^{2} a^{2}-20 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}-4 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}-r^{9} M a^{2}+6 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}+2 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+4 r^{9} M^{3}-4 r^{10} M^{2}+r^{11} M+M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-M^{2} a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}}{4 r M a^{12} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{12} \cos\!\left(\theta\right)^{6}-2 r M a^{12} \cos\!\left(\theta\right)^{8}-8 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{10} \cos\!\left(\theta\right)^{4}+8 r^{2} M^{2} a^{10} \cos\!\left(\theta\right)^{6}+4 r^{2} a^{12} \cos\!\left(\theta\right)^{6}+3 r^{2} a^{12} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{3} M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{10} \cos\!\left(\theta\right)^{4}-20 r^{3} M a^{10} \cos\!\left(\theta\right)^{6}-4 r^{3} M a^{10} \cos\!\left(\theta\right)^{8}+16 r^{3} M^{3} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}-8 r^{3} M^{3} a^{8} \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-40 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-8 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{2}+36 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+16 r^{4} M^{2} a^{8} \cos\!\left(\theta\right)^{6}+6 r^{4} a^{10} \cos\!\left(\theta\right)^{4}+12 r^{4} a^{10} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+24 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{5} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{5} M a^{8} \cos\!\left(\theta\right)^{2}-48 r^{5} M a^{8} \cos\!\left(\theta\right)^{4}-28 r^{5} M a^{8} \cos\!\left(\theta\right)^{6}-2 r^{5} M a^{8} \cos\!\left(\theta\right)^{8}+32 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+16 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-16 r^{5} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{6} \cos\!\left(\theta\right)^{2}-16 r^{5} M^{3} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{6}-8 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2}-56 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-32 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+48 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+60 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+8 r^{6} M^{2} a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} a^{8} \cos\!\left(\theta\right)^{2}+18 r^{6} a^{8} \cos\!\left(\theta\right)^{4}+12 r^{6} a^{8} \cos\!\left(\theta\right)^{6}+r^{6} a^{8} \cos\!\left(\theta\right)^{8}-4 r^{7} M a^{6}+4 r^{7} M a^{6} \sin\!\left(\theta\right)^{2}+24 r^{7} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{7} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-44 r^{7} M a^{6} \cos\!\left(\theta\right)^{2}-60 r^{7} M a^{6} \cos\!\left(\theta\right)^{4}-12 r^{7} M a^{6} \cos\!\left(\theta\right)^{6}-8 r^{7} M^{3} a^{4}+16 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{2}+32 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{4} \sin\!\left(\theta\right)^{4}-32 r^{7} M^{3} a^{4} \cos\!\left(\theta\right)^{2}-8 r^{7} M^{3} a^{4} \cos\!\left(\theta\right)^{4}+20 r^{8} M^{2} a^{4}-24 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-40 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{8} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+72 r^{8} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+28 r^{8} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+r^{8} a^{6}+12 r^{8} a^{6} \cos\!\left(\theta\right)^{2}+18 r^{8} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{8} a^{6} \cos\!\left(\theta\right)^{6}-14 r^{9} M a^{4}+8 r^{9} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{9} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-52 r^{9} M a^{4} \cos\!\left(\theta\right)^{2}-24 r^{9} M a^{4} \cos\!\left(\theta\right)^{4}-16 r^{9} M^{3} a^{2}+16 r^{9} M^{3} a^{2} \sin\!\left(\theta\right)^{2}-16 r^{9} M^{3} a^{2} \cos\!\left(\theta\right)^{2}+28 r^{10} M^{2} a^{2}-16 r^{10} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+32 r^{10} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+3 r^{10} a^{4}+12 r^{10} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{10} a^{4} \cos\!\left(\theta\right)^{4}-16 r^{11} M a^{2}+4 r^{11} M a^{2} \sin\!\left(\theta\right)^{2}-20 r^{11} M a^{2} \cos\!\left(\theta\right)^{2}-8 r^{11} M^{3}+12 r^{12} M^{2}+3 r^{12} a^{2}+4 r^{12} a^{2} \cos\!\left(\theta\right)^{2}-6 r^{13} M+r^{14}+a^{14} \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{r \theta \varphi}$ | $\frac{-8 r M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+4 r M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{3}+r a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{7}-r a^{10} \cos\!\left(\theta\right)^{7}+r a^{10} \cos\!\left(\theta\right)^{9}+3 M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{7}+3 M a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{5}-7 r^{2} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+3 r^{2} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{7}-6 r^{2} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+3 r^{2} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{5}+4 r^{2} M a^{8} \cos\!\left(\theta\right)^{5}-4 r^{2} M a^{8} \cos\!\left(\theta\right)^{7}+4 r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-4 r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+20 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-8 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+16 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-16 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)-4 r^{3} M^{2} a^{6} \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{6} \cos\!\left(\theta\right)^{5}+3 r^{3} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+r^{3} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{7}-3 r^{3} a^{8} \cos\!\left(\theta\right)^{5}+2 r^{3} a^{8} \cos\!\left(\theta\right)^{7}+r^{3} a^{8} \cos\!\left(\theta\right)^{9}-23 r^{4} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-7 r^{4} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-9 r^{4} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-6 r^{4} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+8 r^{4} M a^{6} \cos\!\left(\theta\right)^{3}-4 r^{4} M a^{6} \cos\!\left(\theta\right)^{5}-4 r^{4} M a^{6} \cos\!\left(\theta\right)^{7}-12 r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+4 r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+12 r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-4 r^{5} M^{2} a^{4} \cos\!\left(\theta\right)+28 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+20 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+4 r^{5} M^{2} a^{4} \cos\!\left(\theta\right)^{5}+3 r^{5} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+3 r^{5} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-3 r^{5} a^{6} \cos\!\left(\theta\right)^{3}+3 r^{5} a^{6} \cos\!\left(\theta\right)^{7}+4 r^{6} M a^{4} \cos\!\left(\theta\right)-13 r^{6} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-23 r^{6} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-9 r^{6} M a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)+4 r^{6} M a^{4} \cos\!\left(\theta\right)^{3}-8 r^{6} M a^{4} \cos\!\left(\theta\right)^{5}-12 r^{6} M^{3} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-4 r^{7} M^{2} a^{2} \cos\!\left(\theta\right)+28 r^{7} M^{2} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+4 r^{7} M^{2} a^{2} \cos\!\left(\theta\right)^{3}-r^{7} a^{4} \cos\!\left(\theta\right)+r^{7} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+3 r^{7} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-2 r^{7} a^{4} \cos\!\left(\theta\right)^{3}+3 r^{7} a^{4} \cos\!\left(\theta\right)^{5}+4 r^{8} M a^{2} \cos\!\left(\theta\right)-13 r^{8} M a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-4 r^{8} M a^{2} \cos\!\left(\theta\right)^{3}-r^{9} a^{2} \cos\!\left(\theta\right)+r^{9} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+r^{9} a^{2} \cos\!\left(\theta\right)^{3}}{-4 r M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+4 r M a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{2}+6 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)+16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{5}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4} \sin\!\left(\theta\right)-24 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{3}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{3}+r^{8} a^{4} \sin\!\left(\theta\right)+8 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2} \sin\!\left(\theta\right)-12 r^{9} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{3}+4 r^{10} M^{2} \sin\!\left(\theta\right)+2 r^{10} a^{2} \sin\!\left(\theta\right)+4 r^{10} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-4 r^{11} M \sin\!\left(\theta\right)+r^{12} \sin\!\left(\theta\right)+a^{12} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{\theta t r}$ | $\frac{4 r M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-2 r M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+8 r M^{2} a^{7} \cos\!\left(\theta\right)^{5}-3 M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-2 M a^{9} \cos\!\left(\theta\right)^{7}+6 r^{2} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-r^{2} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+8 r^{2} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-8 r^{2} M^{3} a^{5} \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+4 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+16 r^{3} M^{2} a^{5} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-8 r^{3} M^{2} a^{5} \cos\!\left(\theta\right)^{3}+9 r^{4} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+10 r^{4} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+6 r^{4} M a^{5} \cos\!\left(\theta\right)^{3}+6 r^{4} M a^{5} \cos\!\left(\theta\right)^{5}+16 r^{4} M^{3} a^{3} \cos\!\left(\theta\right)-16 r^{4} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-16 r^{5} M^{2} a^{3} \cos\!\left(\theta\right)-14 r^{5} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-24 r^{5} M^{2} a^{3} \cos\!\left(\theta\right)^{3}+4 r^{6} M a^{3} \cos\!\left(\theta\right)+11 r^{6} M a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+12 r^{6} M a^{3} \cos\!\left(\theta\right)^{3}+24 r^{6} M^{3} a \cos\!\left(\theta\right)-24 r^{7} M^{2} a \cos\!\left(\theta\right)+6 r^{8} M a \cos\!\left(\theta\right)}{-4 r M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+4 r M a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{2}+6 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)+16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{5}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4} \sin\!\left(\theta\right)-24 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{3}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{3}+r^{8} a^{4} \sin\!\left(\theta\right)+8 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2} \sin\!\left(\theta\right)-12 r^{9} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{3}+4 r^{10} M^{2} \sin\!\left(\theta\right)+2 r^{10} a^{2} \sin\!\left(\theta\right)+4 r^{10} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-4 r^{11} M \sin\!\left(\theta\right)+r^{12} \sin\!\left(\theta\right)+a^{12} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{\theta t \theta}$ | $\frac{-8 r M a^{9} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-r M a^{9} \cos\!\left(\theta\right)^{4}-8 r M a^{9} \cos\!\left(\theta\right)^{6}+10 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{2} M^{2} a^{7} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{2} M^{2} a^{7} \cos\!\left(\theta\right)^{2}+30 r^{2} M^{2} a^{7} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{7} \cos\!\left(\theta\right)^{6}-6 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{3} M a^{7} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-16 r^{3} M a^{7} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{7} \cos\!\left(\theta\right)^{6}+28 r^{3} M^{3} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-28 r^{3} M^{3} a^{5} \cos\!\left(\theta\right)^{2}-8 r^{3} M^{3} a^{5} \cos\!\left(\theta\right)^{4}-2 r^{4} M^{2} a^{5}-2 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2}+6 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{4} M^{2} a^{5} \sin\!\left(\theta\right)^{4}+24 r^{4} M^{2} a^{5} \cos\!\left(\theta\right)^{2}+38 r^{4} M^{2} a^{5} \cos\!\left(\theta\right)^{4}+r^{5} M a^{5}+2 r^{5} M a^{5} \sin\!\left(\theta\right)^{2}-6 r^{5} M a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-4 r^{5} M a^{5} \cos\!\left(\theta\right)^{2}-15 r^{5} M a^{5} \cos\!\left(\theta\right)^{4}+12 r^{5} M^{3} a^{3}-12 r^{5} M^{3} a^{3} \sin\!\left(\theta\right)^{2}-36 r^{5} M^{3} a^{3} \cos\!\left(\theta\right)^{2}-14 r^{6} M^{2} a^{3}+2 r^{6} M^{2} a^{3} \sin\!\left(\theta\right)^{2}+26 r^{6} M^{2} a^{3} \cos\!\left(\theta\right)^{2}+4 r^{7} M a^{3}+2 r^{7} M a^{3} \sin\!\left(\theta\right)^{2}-4 r^{7} M a^{3} \cos\!\left(\theta\right)^{2}+12 r^{7} M^{3} a-12 r^{8} M^{2} a+3 r^{9} M a}{4 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{10} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+6 r^{4} a^{8} \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} M^{2} a^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+8 r^{8} M^{2} a^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{8} a^{4}+8 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-12 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+4 r^{10} M^{2}+2 r^{10} a^{2}+4 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-4 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{\theta r \varphi}$ | $\frac{-10 r M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-4 r M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+4 r M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{5}+4 r M^{2} a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{3}+r a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{7}-r a^{10} \cos\!\left(\theta\right)^{7}+r a^{10} \cos\!\left(\theta\right)^{9}+3 M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{7}+3 M a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{5}-7 r^{2} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+3 r^{2} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{7}-6 r^{2} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+3 r^{2} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{5}+4 r^{2} M a^{8} \cos\!\left(\theta\right)^{5}-4 r^{2} M a^{8} \cos\!\left(\theta\right)^{7}+8 r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-8 r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+20 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-6 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+16 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-12 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}-16 r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)-4 r^{3} M^{2} a^{6} \cos\!\left(\theta\right)^{3}+4 r^{3} M^{2} a^{6} \cos\!\left(\theta\right)^{5}+3 r^{3} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}+r^{3} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{7}-3 r^{3} a^{8} \cos\!\left(\theta\right)^{5}+2 r^{3} a^{8} \cos\!\left(\theta\right)^{7}+r^{3} a^{8} \cos\!\left(\theta\right)^{9}-23 r^{4} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-7 r^{4} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-9 r^{4} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-6 r^{4} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{3}+8 r^{4} M a^{6} \cos\!\left(\theta\right)^{3}-4 r^{4} M a^{6} \cos\!\left(\theta\right)^{5}-4 r^{4} M a^{6} \cos\!\left(\theta\right)^{7}-16 r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+16 r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)-4 r^{5} M^{2} a^{4} \cos\!\left(\theta\right)+30 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+28 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+4 r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)+4 r^{5} M^{2} a^{4} \cos\!\left(\theta\right)^{5}+3 r^{5} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}+3 r^{5} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{5}-3 r^{5} a^{6} \cos\!\left(\theta\right)^{3}+3 r^{5} a^{6} \cos\!\left(\theta\right)^{7}+4 r^{6} M a^{4} \cos\!\left(\theta\right)-13 r^{6} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-23 r^{6} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-9 r^{6} M a^{4} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)+4 r^{6} M a^{4} \cos\!\left(\theta\right)^{3}-8 r^{6} M a^{4} \cos\!\left(\theta\right)^{5}-24 r^{6} M^{3} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-4 r^{7} M^{2} a^{2} \cos\!\left(\theta\right)+34 r^{7} M^{2} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+4 r^{7} M^{2} a^{2} \cos\!\left(\theta\right)^{3}-r^{7} a^{4} \cos\!\left(\theta\right)+r^{7} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+3 r^{7} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{3}-2 r^{7} a^{4} \cos\!\left(\theta\right)^{3}+3 r^{7} a^{4} \cos\!\left(\theta\right)^{5}+4 r^{8} M a^{2} \cos\!\left(\theta\right)-13 r^{8} M a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)-4 r^{8} M a^{2} \cos\!\left(\theta\right)^{3}-r^{9} a^{2} \cos\!\left(\theta\right)+r^{9} a^{2} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)+r^{9} a^{2} \cos\!\left(\theta\right)^{3}}{-4 r M a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+4 r M a^{10} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{6}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{5} \cos\!\left(\theta\right)^{2}+6 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}-12 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)+16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{5}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4} \sin\!\left(\theta\right)-24 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{3}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)+8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{3}+r^{8} a^{4} \sin\!\left(\theta\right)+8 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2} \sin\!\left(\theta\right)-12 r^{9} M a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{3}+4 r^{10} M^{2} \sin\!\left(\theta\right)+2 r^{10} a^{2} \sin\!\left(\theta\right)+4 r^{10} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{2}-4 r^{11} M \sin\!\left(\theta\right)+r^{12} \sin\!\left(\theta\right)+a^{12} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{\theta \theta \varphi}$ | $\frac{r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+14 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+8 r M a^{10} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+6 r M a^{10} \cos\!\left(\theta\right)^{8}-2 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-42 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-10 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+20 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+12 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{6} \cos\!\left(\theta\right)^{2}-24 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{6}+26 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+14 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}+6 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{3} M a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+16 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}+6 r^{3} M a^{8} \cos\!\left(\theta\right)^{8}+28 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-16 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-28 r^{3} M^{3} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+24 r^{3} M^{3} a^{6} \cos\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2}-32 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-30 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}-4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{6}-40 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}-24 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{6}-r^{5} M a^{6} \sin\!\left(\theta\right)^{2}+6 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+25 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-2 r^{5} M a^{6} \sin\!\left(\theta\right)^{4}+6 r^{5} M a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}+16 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}-12 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{2}+20 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{5} M^{3} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{5} M^{3} a^{4} \cos\!\left(\theta\right)^{2}+24 r^{5} M^{3} a^{4} \cos\!\left(\theta\right)^{4}+18 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-30 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-6 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}-8 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}-40 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}-6 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+6 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{7} M a^{4} \sin\!\left(\theta\right)^{4}+12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}-8 r^{7} M^{3} a^{2}-4 r^{7} M^{3} a^{2} \sin\!\left(\theta\right)^{2}+16 r^{7} M^{3} a^{2} \cos\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2}+12 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}-8 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}-2 r^{9} M a^{2}-5 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-8 r^{9} M^{3}+8 r^{10} M^{2}-2 r^{11} M}{4 r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-4 r M a^{10} \cos\!\left(\theta\right)^{6}-8 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+4 r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+4 r^{2} a^{10} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{10} \cos\!\left(\theta\right)^{8}+12 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+4 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-12 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-8 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-16 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+8 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+8 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+6 r^{4} a^{8} \cos\!\left(\theta\right)^{4}+8 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+r^{4} a^{8} \cos\!\left(\theta\right)^{8}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+12 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-12 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-24 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-4 r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+4 r^{6} M^{2} a^{4}-8 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-16 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+16 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{2}+12 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+4 r^{6} a^{6} \cos\!\left(\theta\right)^{6}-4 r^{7} M a^{4}+4 r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+12 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-24 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-12 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+8 r^{8} M^{2} a^{2}-8 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+8 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+r^{8} a^{4}+8 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+6 r^{8} a^{4} \cos\!\left(\theta\right)^{4}-8 r^{9} M a^{2}+4 r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-12 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+4 r^{10} M^{2}+2 r^{10} a^{2}+4 r^{10} a^{2} \cos\!\left(\theta\right)^{2}-4 r^{11} M+r^{12}+a^{12} \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{\varphi t \varphi}$ | $\frac{-3 r^{2} M^{2} a^{5} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+6 r^{3} M^{3} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{4} M^{2} a^{3} \sin\!\left(\theta\right)^{2}-3 r^{4} M^{2} a^{3} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{5} M^{3} a \sin\!\left(\theta\right)^{2}+r^{6} M^{2} a \sin\!\left(\theta\right)^{2}}{r M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-r M a^{8} \cos\!\left(\theta\right)^{6}+2 r^{2} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{2} a^{8} \cos\!\left(\theta\right)^{8}+3 r^{3} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{3} M a^{6} \cos\!\left(\theta\right)^{4}-r^{3} M a^{6} \cos\!\left(\theta\right)^{6}+3 r^{4} a^{6} \cos\!\left(\theta\right)^{4}+2 r^{4} a^{6} \cos\!\left(\theta\right)^{6}+3 r^{5} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{2}-3 r^{5} M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{6} a^{4} \cos\!\left(\theta\right)^{2}+3 r^{6} a^{4} \cos\!\left(\theta\right)^{4}-r^{7} M a^{2}+r^{7} M a^{2} \sin\!\left(\theta\right)^{2}-3 r^{7} M a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{2} r^{8} a^{2}+2 r^{8} a^{2} \cos\!\left(\theta\right)^{2}-r^{9} M+\frac{1}{2} r^{10}+\frac{1}{2} a^{10} \cos\!\left(\theta\right)^{8}}$ |
| $R^{\varphi}{}_{\varphi r \theta}$ | $\frac{-\frac{1}{2} r M^{2} a^{8} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+r^{2} M^{3} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}-r^{2} M^{3} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}+\frac{1}{2} r^{3} M^{2} a^{6} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{5}+r^{3} M^{2} a^{6} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)^{3}-r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)-r^{4} M^{3} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+r^{4} M^{3} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)+\frac{1}{2} r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+2 r^{5} M^{2} a^{4} \sin\!\left(\theta\right) \cos\!\left(\theta\right)^{3}+r^{5} M^{2} a^{4} \sin\!\left(\theta\right)^{3} \cos\!\left(\theta\right)-3 r^{6} M^{3} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)+\frac{3}{2} r^{7} M^{2} a^{2} \sin\!\left(\theta\right) \cos\!\left(\theta\right)}{r M a^{10} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-r M a^{10} \cos\!\left(\theta\right)^{6}-2 r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r^{2} M^{2} a^{8} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{4}+r^{2} M^{2} a^{8} \cos\!\left(\theta\right)^{4}+r^{2} a^{10} \cos\!\left(\theta\right)^{6}+\frac{1}{2} r^{2} a^{10} \cos\!\left(\theta\right)^{8}+3 r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+r^{3} M a^{8} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{6}-3 r^{3} M a^{8} \cos\!\left(\theta\right)^{4}-2 r^{3} M a^{8} \cos\!\left(\theta\right)^{6}-4 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}+2 r^{4} M^{2} a^{6} \sin\!\left(\theta\right)^{4} \cos\!\left(\theta\right)^{2}+2 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{2}+2 r^{4} M^{2} a^{6} \cos\!\left(\theta\right)^{4}+\frac{3}{2} r^{4} a^{8} \cos\!\left(\theta\right)^{4}+2 r^{4} a^{8} \cos\!\left(\theta\right)^{6}+\frac{1}{4} r^{4} a^{8} \cos\!\left(\theta\right)^{8}+3 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+3 r^{5} M a^{6} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{4}-3 r^{5} M a^{6} \cos\!\left(\theta\right)^{2}-6 r^{5} M a^{6} \cos\!\left(\theta\right)^{4}-r^{5} M a^{6} \cos\!\left(\theta\right)^{6}+r^{6} M^{2} a^{4}-2 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2}-4 r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}+r^{6} M^{2} a^{4} \sin\!\left(\theta\right)^{4}+4 r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{2}+r^{6} M^{2} a^{4} \cos\!\left(\theta\right)^{4}+r^{6} a^{6} \cos\!\left(\theta\right)^{2}+3 r^{6} a^{6} \cos\!\left(\theta\right)^{4}+r^{6} a^{6} \cos\!\left(\theta\right)^{6}-r^{7} M a^{4}+r^{7} M a^{4} \sin\!\left(\theta\right)^{2}+3 r^{7} M a^{4} \sin\!\left(\theta\right)^{2} \cos\!\left(\theta\right)^{2}-6 r^{7} M a^{4} \cos\!\left(\theta\right)^{2}-3 r^{7} M a^{4} \cos\!\left(\theta\right)^{4}+2 r^{8} M^{2} a^{2}-2 r^{8} M^{2} a^{2} \sin\!\left(\theta\right)^{2}+2 r^{8} M^{2} a^{2} \cos\!\left(\theta\right)^{2}+\frac{1}{4} r^{8} a^{4}+2 r^{8} a^{4} \cos\!\left(\theta\right)^{2}+\frac{3}{2} r^{8} a^{4} \cos\!\left(\theta\right)^{4}-2 r^{9} M a^{2}+r^{9} M a^{2} \sin\!\left(\theta\right)^{2}-3 r^{9} M a^{2} \cos\!\left(\theta\right)^{2}+r^{10} M^{2}+\frac{1}{2} r^{10} a^{2}+r^{10} a^{2} \cos\!\left(\theta\right)^{2}-r^{11} M+\frac{1}{4} r^{12}+\frac{1}{4} a^{12} \cos\!\left(\theta\right)^{8}}$ |